# Data Cleaning Pipeline
**Dissertation:** AI Approaches to Analysing Recruitment Demand: Machine learning insights from European Pharmaceutical Job Postings  
**Author:** Kashmira Bhoir  
**Institution:**  GISMA University of Applied Sciences  
**Date:** 21 June 2026

## Overview
This notebook helps to clean and standardise the combined pharmaceutical job postings dataset making it ready for RQ1, RQ2 and RQ3 analysis.

## Input
- `Combined_Pharma_Jobs.csv` — raw combined dataset (30,243 rows)

## Output
- `Combined_Pharma_Jobs_Cleaned.csv` — cleaned dataset (8,826 rows)

## Cleaning steps are as follows
1. Standardised column names
2. Removed duplicates
3. Cleaned text columns
4. Standardised Post Date formats
5. Parsed Salary → numeric min/max/mid
6. Standardised Job Type variants
7. Standardised Location → country groups
8. Filtered pharma-only jobs by keyword
9. Extracted country from location

Install and Import Libraries

In [6]:
!pip install pandas numpy -q

import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded")


Libraries loaded


In [7]:
file_path = "/content/Combined_Pharma_Jobs.csv"
df = pd.read_csv(
    file_path,
    on_bad_lines = 'skip',
    engine       = 'python'
)
df = df.drop(columns=['Unnamed: 8'], errors='ignore')

print(f"Loaded: {df.shape[0]:,} rows  |  {df.shape[1]} columns")
print(f"\n  Raw columns: {df.columns.tolist()}")

Loaded: 30,243 rows  |  8 columns

  Raw columns: ['Category', 'Company Name', 'Job Description', 'Job Title', 'Job Type', 'Location', 'Post Date', 'Salary Offered']


In [8]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("Column names standardised:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i}. {col}")

Column names standardised:
   1. category
   2. company_name
   3. job_description
   4. job_title
   5. job_type
   6. location
   7. post_date
   8. salary_offered


In [9]:
before = len(df)
df.drop_duplicates(
    subset   = ['job_title', 'company_name', 'location', 'post_date'],
    inplace  = True
)
after = len(df)

print(f"Duplicates removed : {before - after:,}")
print(f"   Rows remaining     : {after:,}")

Duplicates removed : 20,747
   Rows remaining     : 9,496


In [10]:
text_cols = ['job_title', 'company_name', 'category', 'location', 'job_type']

for col in text_cols:
    df[col] = df[col].astype(str).str.strip().str.title()

print("Text columns cleaned")
print(f"\n  Sample job titles:")
print(df['job_title'].value_counts().head(5).to_string())

Text columns cleaned

  Sample job titles:
job_title
Account Manager                 179
Territory Manager               128
Key Account Manager             123
Senior Medical Writer           112
Medical Sales Representative     90


In [11]:
def parse_date(val):
    for fmt in ('%Y-%m-%d', '%d/%m/%y', '%d/%m/%Y', '%m/%d/%Y'):
        try:
            return pd.to_datetime(val, format=fmt)
        except:
            pass
    return pd.NaT

df['post_date'] = df['post_date'].apply(parse_date)

parsed  = df['post_date'].notna().sum()
failed  = df['post_date'].isna().sum()

print(f"Dates parsed successfully : {parsed:,}")
print(f"   Failed (set to NaT)       : {failed:,}")
print(f"\n   Date range: {df['post_date'].min()} → {df['post_date'].max()}")

Dates parsed successfully : 9,496
   Failed (set to NaT)       : 0

   Date range: 2018-03-04 00:00:00 → 2026-01-15 00:00:00


In [23]:
def parse_salary_range(val):
    if pd.isna(val): return None, None
    val_low = str(val).strip().lower()
    vague   = ['competitive','negotiable','neg ','on application',
               'tbc','doe','excellent','attractive','discussed',
               'depending on','uncapped','market rate']
    if any(v in val_low for v in vague): return None, None
    if not re.search(r'\d', val_low):    return None, None
    if re.search(r'p\s*hour|per\s*hour|/hr', val_low): return None, None
    s = re.sub(r'swiss\s*franc|chf|czk|us\$|gbp|eur|€|£|\$', '', val_low)
    s = re.sub(r'up\s*to|approx|from|starting\s*at', '', s)
    s = re.sub(r'(pa|per\s*annum|p\.a\.|pm|per\s*month).*$', '', s)
    s = re.sub(r'(\d),(\d)', r'\1\2', s)
    s = re.sub(r'\+.*$', '', s).strip()
    has_k   = bool(re.search(r'\d\s*k', val_low))
    numbers = re.findall(r'\d+\.?\d*', s)
    if not numbers: return None, None
    nums = [float(n) * (1000 if has_k and float(n) < 1000 else 1)
            for n in numbers]
    nums = [n for n in nums if 10000 <= n <= 500000]
    if not nums: return None, None
    if len(nums) >= 2: return nums[0], nums[1]
    return nums[0], nums[0]

df[['salary_min', 'salary_max']] = df['salary_offered'].apply(
    lambda x: pd.Series(parse_salary_range(x))
)

print(f"\n   Salary min/max sample:")
print(df[['salary_offered','salary_min',
          'salary_max','salary_num']].dropna(
    subset=['salary_num']).head(5).to_string(index=False))


   Salary min/max sample:
salary_offered  salary_min  salary_max  salary_num
         €60k+     60000.0     60000.0     60000.0
     €50k-€70k     50000.0     70000.0     60000.0
         €60k+     60000.0     60000.0     60000.0
         €60k+     60000.0     60000.0     60000.0
         €60k+     60000.0     60000.0     60000.0


In [13]:
job_type_map = {
    'permanent'          : 'Full-Time',
    'full-time'          : 'Full-Time',
    'full time'          : 'Full-Time',
    'contract/interim'   : 'Contract',
    'contract/temp'      : 'Contract',
    'contract'           : 'Contract',
    'part-time'          : 'Part-Time',
    'part time'          : 'Part-Time',
    'temporary/seasonal' : 'Temporary',
    'temporary'          : 'Temporary',
    'remote'             : 'Remote',
    'internship'         : 'Internship',
    'any'                : 'Other',
}

df['job_type_clean'] = (
    df['job_type']
    .str.lower()
    .str.strip()
    .map(job_type_map)
    .fillna('Other')
)

print(f"Job types standardised:")
jt = df['job_type_clean'].value_counts().reset_index()
jt.columns = ['Job Type', 'Count']
jt['Share %'] = (jt['Count'] / len(df) * 100).round(1)
print(jt.to_string(index=False))

Job types standardised:
 Job Type  Count  Share %
Full-Time   8397     88.4
 Contract    876      9.2
   Remote     82      0.9
Part-Time     69      0.7
Temporary     59      0.6
    Other     13      0.1


In [14]:
country_map = {
    'uk'            : 'United Kingdom',
    'london'        : 'United Kingdom',
    'south east'    : 'United Kingdom',
    'north west'    : 'United Kingdom',
    'scotland'      : 'United Kingdom',
    'cambridge'     : 'United Kingdom',
    'm4 corridor'   : 'United Kingdom',
    'oxford'        : 'United Kingdom',
    'manchester'    : 'United Kingdom',
    'germany'       : 'Germany',
    'berlin'        : 'Germany',
    'munich'        : 'Germany',
    'frankfurt'     : 'Germany',
    'hamburg'       : 'Germany',
    'switzerland'   : 'Switzerland',
    'zurich'        : 'Switzerland',
    'basel'         : 'Switzerland',
    'france'        : 'France',
    'paris'         : 'France',
    'netherlands'   : 'Netherlands',
    'amsterdam'     : 'Netherlands',
    'spain'         : 'Spain',
    'italy'         : 'Italy',
    'austria'       : 'Austria',
    'vienna'        : 'Austria',
    'europe'        : 'Europe (General)',
    'remote'        : 'Remote',
}

df['country'] = (
    df['location']
    .str.lower()
    .str.strip()
    .map(country_map)
    .fillna('Other')
)

print(f"Country extracted:")
print(df['country'].value_counts().to_string())

Country extracted:
country
United Kingdom      5448
Europe (General)    2707
Germany              411
Switzerland          349
France               195
Other                176
Spain                 73
Italy                 69
Netherlands           35
Austria               33


In [20]:
pharma_keywords = [
    'Pharmacovigilance', 'Regulatory Affairs', 'Clinical Research',
    'Drug Safety', 'GMP', 'Biotech', 'Pharmaceutical', 'Medical Affairs',
    'Quality Assurance', 'Quality Control', 'Drug Discovery',
    'Clinical Trials', 'Medical Writing', 'Validation', 'Science',
    'Manufacturing', 'Data Management', 'Account Manager', 'Medical Sales'
]

pattern = '|'.join(pharma_keywords)

mask = (
    df['job_title'].str.contains(pattern, case=False, na=False) |
    df['category'].str.contains(pattern, case=False, na=False) |
    df['job_description'].str.contains(pattern, case=False, na=False)
)

before = len(df)
df     = df[mask].copy()
after  = len(df)

print(f"Pharma filter applied")
print(f"   Before : {before:,}")
print(f"   After  : {after:,}")
print(f"   Removed: {before - after:,} non-pharma rows")

Pharma filter applied
   Before : 9,420
   After  : 9,420
   Removed: 0 non-pharma rows


In [22]:

print("  FINAL CLEANED DATASET")
print(f"\n  Rows    : {len(df):,}")
print(f"  Columns : {df.shape[1]}")

print(f"\n  Missing values:")
for col in df.columns:
    nulls = df[col].isnull().sum()
    pct   = (nulls / len(df)) * 100
    flag  = "Error" if pct > 10 else "Success"
    print(f"  {flag} {col:<25} {nulls:>6,}  ({pct:.1f}%)")

print(f"\n  Sample cleaned data (key columns):")
print(df[['job_title', 'category', 'country',
          'job_type_clean', 'salary_offered',
          'salary_num']].head(5).to_string(index=False))

  FINAL CLEANED DATASET

  Rows    : 9,420
  Columns : 11

  Missing values:
  Success category                       0  (0.0%)
  Success company_name                   0  (0.0%)
  Success job_description                0  (0.0%)
  Success job_title                      0  (0.0%)
  Success job_type                       0  (0.0%)
  Success location                       0  (0.0%)
  Success post_date                      0  (0.0%)
  Error salary_offered             2,295  (24.4%)
  Error salary_num                 7,976  (84.7%)
  Success job_type_clean                 0  (0.0%)
  Success country                        0  (0.0%)

  Sample cleaned data (key columns):
                         job_title           category     country job_type_clean salary_offered  salary_num
          Pharmacovigilance Expert Regulatory Affairs      France      Full-Time          €60k+     60000.0
             Drug Safety Associate     Drug Discovery     Germany       Contract    Competitive         NaN
  

In [24]:
output_path = "/content/Combined_Pharma_Jobs_Cleaned.csv"
df.to_csv(output_path, index=False)

print(f"Cleaned file saved → {output_path}")
print(f"   Rows    : {len(df):,}")
print(f"   Columns : {df.shape[1]}")
print(f"\n   Columns in cleaned file:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i}. {col}")

Cleaned file saved → /content/Combined_Pharma_Jobs_Cleaned.csv
   Rows    : 9,420
   Columns : 13

   Columns in cleaned file:
   1. category
   2. company_name
   3. job_description
   4. job_title
   5. job_type
   6. location
   7. post_date
   8. salary_offered
   9. salary_num
   10. job_type_clean
   11. country
   12. salary_min
   13. salary_max
